In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_application_step_fldt(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "ApplicationStepId	string,"
        #     "SourceSystem	string,"
        #     "PortfolioKey	string,"
        #     "PortfolioName	string,"
        #     "IngestionTimestamp	timestamp,"
        #     "BatchId	string,"
        #     "ApplicationStepKey	string,"
        #     "ApplicationKey	string,"
        #     "ApplicationStepCode string,"
        #     "ApplicationStep	string,"
        #     "ApplicationStepDefinition	int,"
        #     "ApplicationFlowStep	int,"
        #     "ApplicationStepIntegrationCode	string,"
        #     "ApplicationStepGroupCode	int,"
        #     "ApplicationStepGroupOwner	int,"
        #     "ApplicationStepReportingName	int,"
        #     "ApplicationStepDuration	bigint,"
        #     "AssignmentDate	timestamp,"
        #     "AssigneeUser	string,"
        #     "AssigneePortfolioGroupCode	string,"
        #     "ExecutionDate	timestamp,"
        #     "ExecutedBy	string,"
        #     "ExecutorPortfolioGroupCode	string,"
        #     "ExecutedActionCode	int,"
        #     "ExecutedAction	int,"
        #     "ExecutedActionName	string,"
        #     "SystemMessage	string,"
        #     "UserMessage	string,"
        #     "IsSystemFlag	boolean,"
        #     "CreditInvolvementStatusCode	int,"
        #     "CreditInvolvementStatus	int,"
        #     "ParentApplicationStepGroupCode	int,"
        #     "ParentApplicationStepGroup	int,"
        #     "ParentApplicationStepGroupOwner	int,"
        #     "ReasonCode	string,"
        #     "Reason	string,"
        #     "IsCurrentStepFlag	string,"
        #     "BusinessDate	date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ApplicationStep"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "fldt",
                )        
        else:
            logging.info("Tables doe not exist, no need to delete entries for business date!")

        # ================== portfolio ===================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== application ==================

        application_extra_filters = None

        if application_extra_filters is not None:
            application_extra_filters.append(date_filter)
            application_final_filters = " AND ".join(application_extra_filters)
        else:
            application_final_filters = date_filter

        application_selected_fields = [
            "id",
            "ApplicationId",
            "ApplicationKey",
            "CurrentStepId",
            "CurrentStepName",
            "CurrentStepDefinitionId",
            "CurrentStepFlowStepId",
            "CurrentStepIntegrationCode",
            "CurrentStepAssignmentDate",
            "AssigneeUserName",
            "AssigneePortfolioGroupCode",
            "CustomerCode",
            "businessdate",
        ]

        application = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_application",
            application_final_filters,
            application_selected_fields,
        )
        # ================== application_metadata ==================

        application_metadata_extra_filters = None

        if application_metadata_extra_filters is not None:
            application_metadata_extra_filters.append(date_filter)
            application_metadata_final_filters = " AND ".join(
                application_metadata_extra_filters
            )
        else:
            application_metadata_final_filters = date_filter

        application_metadata_selected_fields = [
            "SourceApplicationKey",
            "StepGroupingId",
            "StepGroupingOwnerId",
            "StepReportingNameId",
            "StepActiveDuration",
            "CreditInvolvementStatus",
            "ParentStepGroupingId",
            "ParentStepGroupingOwnerId",
            "CreditInvolvementStatusDescription",
            "ParentStepGroupingDescription",
            "IsActive",
        ]

        application_metadata = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_applicationmetadata",
            application_metadata_final_filters,
            application_metadata_selected_fields,
        )
        # ================== completed_application_step ==================

        completed_application_step_extra_filters = None

        if completed_application_step_extra_filters is not None:
            completed_application_step_extra_filters.append(date_filter)
            completed_application_step_final_filters = " AND ".join(
                completed_application_step_extra_filters
            )
        else:
            completed_application_step_final_filters = date_filter

        completed_application_step_selected_fields = [
            "ReasonCode",
            "stepid",
            "ApplicationId",
            "StepName",
            "StepDefinitionId",
            "FlowStepId",
            "StepIntegrationCode",
            "AssigneeUserName",
            "ExecutionDate",
            "ExecutedByUserId",
            "ExecutorPortfolioGroupCode",
            "ExecutedActionId",
            "ExecutedActionName",
            "executedactionsystemname",
            "SystemMessage",
            "UserMessage",
            "ReasonCodeDescription",
            "IsSystem",
        ]

        completed_application_step = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_completedapplicationstep",
            completed_application_step_final_filters,
            completed_application_step_selected_fields,
        )
        # ================== application_step_metadata ==================

        application_step_metadata_extra_filters = None

        if application_step_metadata_extra_filters is not None:
            application_step_metadata_extra_filters.append(date_filter)
            application_step_metadata_final_filters = " AND ".join(
                application_step_metadata_extra_filters
            )
        else:
            application_step_metadata_final_filters = date_filter

        application_step_metadata_selected_fields = [
            "StepGroupingId",
            "StepGroupingOwnerId",
            "StepReportingNameId",
            "StepDuration",
            "ParentStepGroupingDescription",
            "CreditInvolvementStatus",
            "creditinvolvementstatusdescription",
            "ParentStepGroupingId",
            "ParentStepGroupingOwnerId",
            "SourceStepId",
        ]

        application_step_metadata = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_applicationstepmetadata",
            application_step_metadata_final_filters,
            application_step_metadata_selected_fields,
        )

        # ================== application_step_app_df ==================

        application_step_primary_key_fields = [
            application.CurrentStepId,
            application.ApplicationKey,
            lit("fldt"),
        ]

        application_step_app_df = (
            application.join(
                application_metadata,
                (
                    application_metadata["SourceApplicationKey"]
                    == application["ApplicationKey"]
                )
                & (application_metadata["IsActive"] == 1),
                "left",
            )
            .join(
                portfolio,
                (substring(application.CustomerCode, 1, 3) == portfolio.Prefix),
                "left",
            )
            .withColumn(
                "ApplicationIdKey",
                sha2(concat_ws(";", *application_step_primary_key_fields), 256),
            )
            .withColumn(
                "ApplicationStepKey",
                concat_ws(";", application.CurrentStepId, application.ApplicationKey),
            )
            .withColumn(
                "AssigneePortfolioGroupCode",
                coalesce(application["AssigneePortfolioGroupCode"], lit("-1")),
            )
        )

        application_step_app_df = application_step_app_df.select(
            application_step_app_df.ApplicationIdKey.cast("string").alias(
                "ApplicationStepId"
            ),
            lit("fldt").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchID"),
            application_step_app_df.ApplicationStepKey.alias("ApplicationStepkey"),
            application_step_app_df.ApplicationKey.alias("ApplicationKey"),
            application_step_app_df.CurrentStepId.alias("ApplicationStepCode"),
            application_step_app_df.CurrentStepName.alias("ApplicationStep"),
            application_step_app_df.CurrentStepDefinitionId.alias(
                "ApplicationStepDefinition"
            ),
            application_step_app_df.CurrentStepFlowStepId.alias("ApplicationFlowStep"),
            application_step_app_df.CurrentStepIntegrationCode.alias(
                "ApplicationStepIntegrationCode"
            ),
            application_step_app_df.StepGroupingId.alias("ApplicationStepGroupCode"),
            application_step_app_df.StepGroupingOwnerId.alias(
                "ApplicationStepGroupOwner"
            ),
            application_step_app_df.StepReportingNameId.alias(
                "ApplicationStepReportingName"
            ),
            application_step_app_df.StepActiveDuration.alias("ApplicationStepDuration"),
            application_step_app_df.CurrentStepAssignmentDate.alias("AssignmentDate"),
            application_step_app_df.AssigneeUserName.alias("AssigneeUser"),
            application_step_app_df.AssigneePortfolioGroupCode,
            lit(None).alias("ExecutionDate"),
            lit(None).alias("ExecutedBy"),
            lit(None).alias("ExecutorPortfolioGroupCode"),
            lit(None).alias("ExecutedActionCode"),
            lit(None).alias("ExecutedAction"),
            lit(None).alias("ExecutedActionName"),
            lit(None).alias("SystemMessage"),
            lit(None).alias("UserMessage"),
            lit(None).alias("IsSystemFlag"),
            application_step_app_df.CreditInvolvementStatus.alias(
                "CreditInvolvementStatusCode"
            ),
            application_step_app_df.CreditInvolvementStatusDescription.cast(
                "int"
            ).alias("CreditInvolvementStatus"),
            application_step_app_df.ParentStepGroupingId.cast("int").alias(
                "ParentApplicationStepGroupCode"
            ),
            application_step_app_df.ParentStepGroupingDescription.cast("int").alias(
                "ParentApplicationStepGroup"
            ),
            application_step_app_df.ParentStepGroupingOwnerId.alias(
                "ParentApplicationStepGroupOwner"
            ),
            lit(None).alias("ReasonCode"),
            lit(None).alias("Reason"),
            lit("1").alias("IsCurrentStepFlag"),
            application.businessdate.alias("BusinessDate"),
        )

        # ================== application_step_cas_df ==================

        application_step_primary_key_fields = [
            completed_application_step.stepid,
            lit("fldt"),
        ]

        application_completed_application_step_join_condition = (
            completed_application_step["ApplicationId"] == application["id"]
        )
        CompletedApplicationStep_ApplicationStepMetadata_join_condition = (
            completed_application_step["stepid"]
            == application_step_metadata["SourceStepId"]
        )

        application_step_cas_df = (
            application.join(
                completed_application_step,
                application_completed_application_step_join_condition,
                "inner",
            )
            .join(
                application_step_metadata,
                CompletedApplicationStep_ApplicationStepMetadata_join_condition,
                "left",
            )
            .join(
                portfolio,
                (substring(application.CustomerCode, 1, 3) == portfolio.Prefix),
                "left",
            )
            .withColumn(
                "ApplicationIdKey",
                sha2(concat_ws(";", *application_step_primary_key_fields), 256),
            )
            .withColumn(
                "AssigneePortfolioGroupCode",
                coalesce(application["AssigneePortfolioGroupCode"], lit("-1")),
            )
            .withColumn(
                "ExecutorPortfolioGroupCode",
                coalesce(
                    completed_application_step["ExecutorPortfolioGroupCode"], lit("-1")
                ),
            )
            .withColumn(
                "AssigneeUserName_cas", completed_application_step["AssigneeUserName"]
            )
            .withColumn(
                "ApplicationId_cas", completed_application_step["ApplicationId"]
            )
        )

        application_step_cas_df = application_step_cas_df.select(
            application_step_cas_df.ApplicationIdKey.cast("string").alias(
                "ApplicationStepId"
            ),
            lit("fldt").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchID"),
            completed_application_step.stepid.alias("ApplicationStepKey"),
            application.ApplicationKey.alias("ApplicationKey"),
            application_step_cas_df.stepid.alias("ApplicationStepCode"),
            application_step_cas_df.StepName.alias("ApplicationStep"),
            application_step_cas_df.StepDefinitionId.alias("ApplicationStepDefinition"),
            application_step_cas_df.FlowStepId.alias("ApplicationFlowStep"),
            application_step_cas_df.StepIntegrationCode.alias(
                "ApplicationStepIntegrationCode"
            ),
            application_step_cas_df.StepGroupingId.alias("ApplicationStepGroupCode"),
            application_step_cas_df.StepGroupingOwnerId.alias(
                "ApplicationStepGroupOwner"
            ),
            application_step_cas_df.StepReportingNameId.alias(
                "ApplicationStepReportingName"
            ),
            application_step_cas_df.StepDuration.alias("ApplicationStepDuration"),
            application_step_cas_df.CurrentStepAssignmentDate.alias("AssignmentDate"),
            application_step_cas_df.AssigneeUserName_cas.alias("AssigneeUser"),
            application_step_cas_df.AssigneePortfolioGroupCode,
            application_step_cas_df.ExecutionDate.alias("ExecutionDate"),
            application_step_cas_df.ExecutedByUserId.alias("ExecutedBy"),
            application_step_cas_df.ExecutorPortfolioGroupCode,
            application_step_cas_df.ExecutedActionId.alias("ExecutedActionCode"),
            application_step_cas_df.executedactionsystemname.cast("int").alias(
                "ExecutedAction"
            ),
            application_step_cas_df.ExecutedActionName.alias("ExecutedActionName"),
            application_step_cas_df.SystemMessage.alias("SystemMessage"),
            application_step_cas_df.UserMessage.alias("UserMessage"),
            application_step_cas_df.IsSystem.alias("IsSystemFlag"),
            application_step_cas_df.CreditInvolvementStatus.alias(
                "CreditInvolvementStatusCode"
            ),
            application_step_cas_df.creditinvolvementstatusdescription.cast(
                "int"
            ).alias("CreditInvolvementStatus"),
            application_step_cas_df.ParentStepGroupingId.cast("int").alias(
                "ParentApplicationStepGroupCode"
            ),
            application_step_cas_df.ParentStepGroupingDescription.cast("int").alias(
                "ParentApplicationStepGroup"
            ),
            application_step_cas_df.ParentStepGroupingOwnerId.alias(
                "ParentApplicationStepGroupOwner"
            ),
            application_step_cas_df.ReasonCode.alias("ReasonCode"),
            application_step_cas_df.ReasonCodeDescription.cast("string").alias(
                "Reason"
            ),
            lit("0").alias("IsCurrentStepFlag"),
            application.businessdate.alias("BusinessDate"),
        )
        # ================== Append ==================
        application_step_df = application_step_cas_df.union(application_step_app_df)

        application_step_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_application_step_fldt(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)